# Асинхронный веб-краулер. День 1

## Установка зависимостей

In [1]:
%pip install aiohttp aiofiles

Note: you may need to restart the kernel to use updated packages.


### Импорты
Все библиотеки, которые используются в ноутбуке, собраны здесь.

In [ ]:
import asyncio
import aiohttp
import logging
import time

### Настройка логирования

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("crawler")

## Класс AsyncCrawler
Сессия создаётся один раз и переиспользуется для всех запросов.
Семафор ограничивает число одновременных запросов значением max_concurrent.

In [4]:
class AsyncCrawler:
    def __init__(
        self,
        max_concurrent: int = 10,
        connect_timeout: float = 5,
        read_timeout: float = 10,
    ):
        self.max_concurrent = max_concurrent
        self.semaphore = asyncio.Semaphore(max_concurrent)

        timeout = aiohttp.ClientTimeout(
            connect=connect_timeout,
            sock_read=read_timeout,
        )
        # используется системный поиск адресов, чтобы краулер работал через VPN
        connector = aiohttp.TCPConnector(
            limit=max_concurrent,
            resolver=aiohttp.ThreadedResolver(),
        )

        self.session = aiohttp.ClientSession(
            timeout=timeout,
            connector=connector,
        )

    async def fetch_url(self, url: str) -> str | None:
        async with self.semaphore:
            logger.info("Начало загрузки: %s", url)
            try:
                async with self.session.get(url) as response:
                    response.raise_for_status()
                    text = await response.text()
                    logger.info(
                        "Успешно: %s (статус %s, %d символов)",
                        url, response.status, len(text),
                    )
                    return text
            except aiohttp.ClientResponseError as e:
                logger.error("%s, статус %s: %s", type(e).__name__, e.status, url)
            except asyncio.TimeoutError as e:
                logger.error("%s, таймаут: %s", type(e).__name__, url)
            except aiohttp.ClientError as e:
                logger.error("%s, сетевая ошибка: %s", type(e).__name__, url)
            return None

    async def fetch_urls(self, urls: list[str]) -> dict[str, str]:
        tasks = [self.fetch_url(url) for url in urls]
        pages = await asyncio.gather(*tasks)
        return {url: page for url, page in zip(urls, pages) if page is not None}

    async def close(self):
        await self.session.close()

## Демонстрация 1 дня

In [ ]:
DEMO_URLS = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/get",
    "https://httpbin.org/html",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/3",
    "https://httpbin.org/status/404",
]


async def run_sequential(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    results = {}
    start = time.perf_counter()
    try:
        for url in urls:
            page = await crawler.fetch_url(url)
            if page is not None:
                results[url] = page
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


async def run_parallel(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        results = await crawler.fetch_urls(urls)
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


def print_report(title: str, urls: list[str], results: dict[str, str], elapsed: float):
    print(f"\n{title}")
    for url in urls:
        mark = "✅" if url in results else "❌"
        print(f"  {mark} {url}")
    print(f"  Загружено {len(results)} из {len(urls)} за {elapsed:.2f} сек")


async def main():
    seq_results, seq_time = await run_sequential(DEMO_URLS)
    par_results, par_time = await run_parallel(DEMO_URLS)

    print_report("Последовательная загрузка", DEMO_URLS, seq_results, seq_time)
    print_report("Параллельная загрузка", DEMO_URLS, par_results, par_time)
    print(f"\nПараллельно быстрее в {seq_time / par_time:.1f} раза")


await main()

18:19:23 [INFO] Начало загрузки: https://example.com
18:19:24 [INFO] Успешно: https://example.com (статус 200, 577 символов)
18:19:24 [INFO] Начало загрузки: https://www.python.org
18:19:24 [INFO] Успешно: https://www.python.org (статус 200, 51909 символов)
18:19:24 [INFO] Начало загрузки: https://httpbin.org/get
18:19:25 [INFO] Успешно: https://httpbin.org/get (статус 200, 322 символов)
18:19:25 [INFO] Начало загрузки: https://httpbin.org/html
18:19:25 [INFO] Успешно: https://httpbin.org/html (статус 200, 3739 символов)
18:19:25 [INFO] Начало загрузки: https://httpbin.org/delay/1
18:19:27 [INFO] Успешно: https://httpbin.org/delay/1 (статус 200, 372 символов)
18:19:27 [INFO] Начало загрузки: https://httpbin.org/delay/2
18:19:29 [INFO] Успешно: https://httpbin.org/delay/2 (статус 200, 372 символов)
18:19:29 [INFO] Начало загрузки: https://httpbin.org/delay/3
18:19:32 [INFO] Успешно: https://httpbin.org/delay/3 (статус 200, 372 символов)
18:19:32 [INFO] Начало загрузки: https://httpbin.o


Последовательная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 9.50 сек

Параллельная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 3.79 сек

Параллельно быстрее в 2.5 раза


## Тестирование 1 дня

In [6]:
async def test_valid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://example.com",
            "https://httpbin.org/html",
        ])
    finally:
        await crawler.close()
    assert len(results) == 2, f"ожидали 2 страницы, получили {len(results)}"
    assert "Example Domain" in results["https://example.com"], "не тот текст страницы"


async def test_invalid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://this-site-does-not-exist-12345.com",
            "https://httpbin.org/status/404",
        ])
    finally:
        await crawler.close()
    assert results == {}, f"ожидали пустой результат, получили {list(results)}"


async def test_timeout():
    crawler = AsyncCrawler(read_timeout=1)
    start = time.perf_counter()
    try:
        page = await crawler.fetch_url("https://httpbin.org/delay/5")
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start
    assert page is None, "страница не должна была загрузиться"
    assert elapsed < 4, f"таймаут не сработал, ждали {elapsed:.1f} сек"


async def test_parallel_faster():
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/delay/3",
    ]
    _, seq_time = await run_sequential(urls)
    _, par_time = await run_parallel(urls)
    assert par_time < seq_time * 0.75, (
        f"параллельно {par_time:.1f} сек, последовательно {seq_time:.1f} сек"
    )


async def run_all_tests():
    tests = [test_valid_urls, test_invalid_urls, test_timeout, test_parallel_faster]
    logger.setLevel(logging.WARNING)
    passed = 0
    try:
        for test in tests:
            try:
                await test()
                print(f"✅ {test.__name__}")
                passed += 1
            except AssertionError as e:
                print(f"❌ {test.__name__}: {e}")
    finally:
        logger.setLevel(logging.INFO)
    print(f"\nПройдено {passed} из {len(tests)}")


await run_all_tests()

18:19:37 [ERROR] ClientConnectorDNSError, сетевая ошибка: https://this-site-does-not-exist-12345.com


✅ test_valid_urls


18:19:38 [ERROR] ClientResponseError, статус 404: https://httpbin.org/status/404


✅ test_invalid_urls


18:19:39 [ERROR] SocketTimeoutError, таймаут: https://httpbin.org/delay/5


✅ test_timeout
✅ test_parallel_faster

Пройдено 4 из 4
